# Laplace Approximation: Gaussian Mixture

This example is part of the *Universal Inference* section, which explores running inference in a broad class of models by approximating every continuous latent variable with a Gaussian. Real models rarely consist of unconstrained continuous variables only. A Gaussian mixture has two other kinds:

- **discrete variables**, the component assignments $z_i$, which have a finite number of values. They need no approximation: their beliefs are normalised exactly, and expectations over them are exact sums.
- **constrained variables**, the component precisions, which must be positive. Here we make them Gaussian in an *unconstrained* space by modelling the log-precisions $s_k$, so that the precision is $e^{s_k} > 0$ for every value of $s_k$. (The *Constrained Variables* example of this section shows that the fallback can also do this by itself, from the support of a prior.)

The model is

$$\begin{aligned}
\pi &\sim \mathrm{Dirichlet}(1, \ldots, 1),\\
m &\sim \mathcal{N}(0, 100 I),\qquad s \sim \mathcal{N}(0, 4 I),\\
z_i &\sim \mathrm{Categorical}(\pi),\\
y_i &\sim \mathcal{N}\left(m_{z_i}, e^{-s_{z_i}}\right).
\end{aligned}$$

The Dirichlet-Categorical pair is conjugate, so RxInfer keeps its exact rules for it. Everything else is handled by `fallback = LaplaceApproximation()`: Laplace approximations for $m$ and $s$, exact beliefs for the $z_i$, and proper variational messages, expectations of the likelihood's log-density over the other variables' beliefs. We compare the result with RxInfer's hand-derived conjugate solution for the same kind of model.

In [ ]:
using RxInfer, BayesBase, Distributions, LinearAlgebra
using StatsFuns, StableRNGs, Plots

## The likelihood node

The mixture likelihood is a node of its own with the interfaces `out`, `z`, `m` and `s`, and no message update rules. Its `logpdf` accepts the assignment either as a component index or as a vector of weights; we need the latter only for a comparison at the end.

In [ ]:
struct LogPrecisionMixture{Z, M, S} <: ContinuousUnivariateDistribution
    z::Z
    m::M
    s::S
end

component_weights(z::Integer, K) = (1:K) .== z
component_weights(z::AbstractVector, K) = z

function BayesBase.logpdf(d::LogPrecisionMixture, y)
    w = component_weights(d.z, length(d.m))
    return sum(w[k] * (d.s[k] - log2π - exp(d.s[k]) * (y - d.m[k])^2) / 2 for k in eachindex(d.m))
end
BayesBase.insupport(::LogPrecisionMixture, y) = true

@define_factor_node(node = LogPrecisionMixture, type = Stochastic, interfaces = [:out, :z, :m, :s])

## Data

In [ ]:
rng = StableRNG(42)
N, K = 300, 2
true_weights, true_means, true_sds = [0.4, 0.6], [-2.0, 2.0], [0.5, 1.0]
z_true = rand(rng, Categorical(true_weights), N)
y = [rand(rng, Normal(true_means[z], true_sds[z])) for z in z_true]

histogram(y; bins = 40, normalize = :pdf, color = :gray, alpha = 0.5, label = "data", xlabel = "y")

## Inference

The model is written with vectors $m$ and $s$ of length $K$, so the same code works for any number of components. Mean-field inference is iterative and needs a starting point. The fallback starts every variable from its prior, except where we say otherwise: the prior of $m$ is symmetric in the components, so we give $m$ initial means that differ, to break the symmetry between the components, as in any variational treatment of mixtures.

In [ ]:
@model function laplace_mixture(y, K)
    π ~ Dirichlet(ones(K))
    m ~ MvNormalMeanCovariance(zeros(K), 100.0 * diageye(K))
    s ~ MvNormalMeanCovariance(zeros(K), 4.0 * diageye(K))
    for i in eachindex(y)
        z[i] ~ Categorical(π)
        y[i] ~ LogPrecisionMixture(z[i], m, s)
    end
end

mean_field = @constraints begin
    q(π, m, s, z) = q(π)q(m)q(s)q(z)
end

# The initial means differ to break the symmetry between the components; the priors start the rest
initial_means = @initialization begin
    q(m) = MvNormalMeanCovariance([-1.0, 1.0], diageye(K))
end

n_iterations = 30

result = infer(
    model = laplace_mixture(K = K),
    data = (y = y,),
    constraints = mean_field,
    initialization = initial_means,
    iterations = n_iterations,
    returnvars = (m = KeepEach(), s = KeepEach(), π = KeepLast()),
    fallback = LaplaceApproximation(),
)

q_m, q_s, q_π = result.posteriors[:m][end], result.posteriors[:s][end], result.posteriors[:π]

We monitor convergence through the posterior means of the components (see the end of this notebook for the free energy).

In [ ]:
plot(1:n_iterations, reduce(hcat, mean.(result.posteriors[:m]))'; label = ["E[m₁]" "E[m₂]"], xlabel = "iteration", ylabel = "posterior mean", marker = :circle)
@assert maximum(abs, mean(result.posteriors[:m][end]) - mean(result.posteriors[:m][end - 5])) < 1e-6 # hide

## Reference: the conjugate model

RxInfer has exact variational rules for a Gaussian mixture with Gamma-distributed precisions, the `NormalMixture` node. Its priors differ slightly from ours (a Gamma prior on the precision instead of a Gaussian prior on the log-precision), so we expect close but not identical results.

In [ ]:
@model function conjugate_mixture(y, K)
    local m
    local w
    π ~ Dirichlet(ones(K))
    for k in 1:K
        m[k] ~ NormalMeanVariance(0.0, 100.0)
        w[k] ~ GammaShapeRate(1.0, 1.0)
    end
    for i in eachindex(y)
        z[i] ~ Categorical(π)
        y[i] ~ NormalMixture(switch = z[i], m = m, p = w)
    end
end

conjugate_initialization = @initialization begin
    q(π) = Dirichlet(ones(K))
    q(m) = [NormalMeanVariance(-1.0, 1.0), NormalMeanVariance(1.0, 1.0)]
    q(w) = [GammaShapeRate(1.0, 1.0), GammaShapeRate(1.0, 1.0)]
end

result_conjugate = infer(
    model = conjugate_mixture(K = K),
    data = (y = y,),
    constraints = MeanField(),
    initialization = conjugate_initialization,
    iterations = n_iterations,
    returnvars = KeepLast(),
)

We report the component standard deviation at the posterior mean of the log-precision, $e^{-\mathbb{E}[s_k]/2}$, and for the conjugate model at the posterior mean of the precision, $\mathbb{E}[w_k]^{-1/2}$.

In [ ]:
row(name, means, sds, component_sds, weights) = println(rpad(name, 22),
    "m = ", round.(means; digits = 3), "  sd(m) = ", round.(sds; digits = 3),
    "  component sd = ", round.(component_sds; digits = 3), "  π = ", round.(weights; digits = 3))

laplace_sds = exp.(-mean(q_s) ./ 2)
conjugate_means = mean.(result_conjugate.posteriors[:m])
conjugate_sds = 1 ./ sqrt.(mean.(result_conjugate.posteriors[:w]))

row("truth", true_means, [0.0, 0.0], true_sds, true_weights)
row("empirical (true z)", [sum(y[z_true .== k]) / count(z_true .== k) for k in 1:K], [0.0, 0.0], [std(y[z_true .== k]) for k in 1:K], [count(z_true .== k) / N for k in 1:K])
row("Laplace + exact z", mean(q_m), sqrt.(diag(cov(q_m))), laplace_sds, mean(q_π))
row("conjugate VMP", conjugate_means, std.(result_conjugate.posteriors[:m]), conjugate_sds, mean(result_conjugate.posteriors[:π]))
@assert maximum(abs, mean(q_m) - conjugate_means) < 0.02 # hide
@assert maximum(abs, laplace_sds - conjugate_sds) < 0.02 # hide
@assert maximum(abs, mean(q_π) - mean(result_conjugate.posteriors[:π])) < 0.01 # hide

In [ ]:
ys = range(minimum(y) - 1, maximum(y) + 1; length = 400)
density_laplace = [sum(mean(q_π)[k] * pdf(Normal(mean(q_m)[k], laplace_sds[k]), v) for k in 1:K) for v in ys]
density_conjugate = [sum(mean(result_conjugate.posteriors[:π])[k] * pdf(Normal(conjugate_means[k], conjugate_sds[k]), v) for k in 1:K) for v in ys]

histogram(y; bins = 40, normalize = :pdf, color = :gray, alpha = 0.4, label = "data", xlabel = "y")
plot!(ys, density_laplace; color = :red, linewidth = 2, label = "Laplace + exact z")
plot!(ys, density_conjugate; color = :blue, linewidth = 2, linestyle = :dash, label = "conjugate VMP")

## Plug-in vs expected messages

How much does computing proper expectations matter? RxInfer's built-in `NodeFunctionRuleFallback` replaces the expectation over the other variables with a plug-in: it evaluates the log-density at their *means*. That ignores their uncertainty, and with its default it breaks for discrete variables, because the mean of a categorical distribution over $\{1, \ldots, K\}$ is a number between $1$ and $K$, not a component index. To make it work at all, we pass it an `extract` function that collapses a categorical to its vector of probabilities.

Anything set explicitly wins over the keyword, so we keep `fallback = LaplaceApproximation()`, which still provides the Laplace projections and the initialisation, and replace only its rule fallback through `options`. With $N = 300$ observations the beliefs of $m$ and $s$ are narrow and the two fallbacks agree closely; the difference grows as the data become scarcer, because the plug-in ignores the uncertainty of the other variables. In particular, it ignores $\mathrm{Var}[m_k]$ when it updates $s_k$, and so tends to underestimate the component spread.

In [ ]:
plugin_extract(q) = q isa Categorical ? probs(q) : mean(q)

function component_sds(y; options = nothing)
    result = infer(
        model = laplace_mixture(K = K),
        data = (y = y,),
        constraints = mean_field,
        initialization = initial_means,
        iterations = n_iterations,
        returnvars = KeepLast(),
        fallback = LaplaceApproximation(),
        options = options,
    )
    return exp.(-mean(result.posteriors[:s]) ./ 2)
end

for n in (300, 40, 15)
    expected_sds = component_sds(y[1:n])
    plugin_sds = component_sds(y[1:n]; options = (rulefallback = NodeFunctionRuleFallback(plugin_extract),))
    println("N = ", lpad(n, 3), ":  component sd, expected = ", round.(expected_sds; digits = 3), ",  plug-in = ", round.(plugin_sds; digits = 3))
    n == 300 && @assert maximum(abs, expected_sds - plugin_sds) < 0.01 # hide
    n == 15 && @assert all(plugin_sds .< expected_sds) # hide
end

## Exact vs approximate, and what is still missing

- **What worked.** Gaussian approximations for the continuous parameters, a Gaussian in log-space for the positive precisions, and exact beliefs for the discrete assignments together recover the mixture, and agree with RxInfer's hand-derived conjugate solution. No message update rule was derived by hand, and the only setting the model needed was the initial means that break the symmetry of the components.
- **What the keyword brings.** The variational messages are expectations, exact sums over the categorical beliefs of $z_i$ and quadrature over the Gaussian beliefs of $m$ and $s$, so they account for the uncertainty that the plug-in ignores. The beliefs of the $z_i$ are normalised exactly.
- **What is still missing.**
  - The free energy. The fallback computes the average energies of the likelihood nodes, but the energy rule of the Gaussian prior node of $m$ expects RxInfer's own Gaussian types, and the Laplace projection of a vector variable returns a Distributions.jl `MvNormal`.
  - The rule fallback does not apply to RxInfer's mixture nodes (`NormalMixture`, `Mixture`), because they have interface groups. We had to write the mixture likelihood with an explicit assignment variable.